In [1]:
import ast
import gradio as gr

# ------------------------------------------------------------
# AST ANALYZER
# ------------------------------------------------------------

class FunctionAnalyzer(ast.NodeVisitor):
    def __init__(self):
        self.functions = []

    def visit_FunctionDef(self, node):
        function_data = {
            "name": node.name,
            "args": [],
            "returns": None,
            "has_return": False
        }

        # Arguments
        for arg in node.args.args:
            function_data["args"].append(arg.arg)

        # Detect return annotation
        if node.returns:
            function_data["returns"] = ast.unparse(node.returns)

        # Detect if function has return statement
        for child in ast.walk(node):
            if isinstance(child, ast.Return):
                function_data["has_return"] = True

        self.functions.append(function_data)

        self.generic_visit(node)


# ------------------------------------------------------------
# DOCSTRING GENERATOR
# ------------------------------------------------------------

def generate_docstring(func, style="google"):
    name = func["name"]
    args = func["args"]
    returns = func["returns"]
    has_return = func["has_return"]

    summary = f"{name.replace('_', ' ').capitalize()} function."

    if style == "google":
        doc = f'"""{summary}\n\n'

        if args:
            doc += "Args:\n"
            for arg in args:
                doc += f"    {arg} (type): Description of {arg}.\n"
            doc += "\n"

        if has_return:
            return_type = returns if returns else "type"
            doc += "Returns:\n"
            doc += f"    {return_type}: Description of return value.\n"

        doc += '"""'

    elif style == "numpy":
        doc = f'"""{summary}\n\n'

        if args:
            doc += "Parameters\n"
            doc += "----------\n"
            for arg in args:
                doc += f"{arg} : type\n"
                doc += f"    Description of {arg}.\n"
            doc += "\n"

        if has_return:
            return_type = returns if returns else "type"
            doc += "Returns\n"
            doc += "-------\n"
            doc += f"{return_type}\n"
            doc += "    Description of return value.\n"

        doc += '"""'

    return doc


# ------------------------------------------------------------
# MAIN PROCESSING FUNCTION
# ------------------------------------------------------------

def process_code(code, style):
    try:
        tree = ast.parse(code)
        analyzer = FunctionAnalyzer()
        analyzer.visit(tree)

        if not analyzer.functions:
            return "No functions detected."

        output = ""

        for func in analyzer.functions:
            docstring = generate_docstring(func, style)
            output += f"\nFunction: {func['name']}\n"
            output += docstring
            output += "\n\n"

        return output

    except Exception as e:
        return f"Error parsing code: {str(e)}"


# ------------------------------------------------------------
# GRADIO UI
# ------------------------------------------------------------

with gr.Blocks(title="Doc-Genie") as app:
    gr.Markdown("# 🧠 Doc-Genie: Professional Python Docstring Generator")

    code_input = gr.Code(label="Paste Python Code Here", language="python")
    style_choice = gr.Radio(["google", "numpy"], value="google", label="Docstring Style")
    generate_btn = gr.Button("Generate Docstrings")
    output_box = gr.Textbox(label="Generated Docstrings", lines=20)

    generate_btn.click(
        fn=process_code,
        inputs=[code_input, style_choice],
        outputs=output_box
    )

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bd021da89abadec64c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
